# D4_03 선형 vs ML 회귀

> **오늘 질문:** 직선으로 못 잡는 관계를 트리 모델은 잡을 수 있을까? 대신 무엇을 잃을까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False   # 음수 부호 깨짐 방지
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

cc = pd.read_csv("../../data/concrete.csv")
X = cc.drop(columns="strength")
y = cc["strength"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. 재령과 강도의 관계 ✍️

강도는 재령 초기에 급하게 오르고, 이후 완만해집니다. 직선으로 표현하기 어려운 모양입니다.

In [ ]:
# 힌트: sns.scatterplot(data=표, x=..., y=..., alpha=0.5)

## 2. 교차검증 준비 ✍️

Train 안에서 5-Fold로 나눠 평가합니다. MAE는 scikit-learn에서 '클수록 좋게' 맞추려고 음수로 계산(`neg_mean_absolute_error`)하므로, 부호를 바꿔 읽습니다.

> 변수 이름: `cv`

In [ ]:
# 힌트: KFold(5, shuffle=True, random_state=42), cross_validate(모델, X, y, cv=cv, scoring=[...])

## 3. 세 모델 비교 📋

같은 교차검증으로 선형회귀, 의사결정나무, 랜덤포레스트를 비교합니다. (10초 정도)

In [ ]:
models = {
    "선형회귀": LinearRegression(),
    "의사결정나무": DecisionTreeRegressor(random_state=42),
    "랜덤포레스트": RandomForestRegressor(n_estimators=200, random_state=42),
}
rows = []
for name, model in models.items():
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=["neg_mean_absolute_error", "r2"])
    rows.append({"모델": name, "교차검증 MAE": -r["test_neg_mean_absolute_error"].mean(), "교차검증 R²": r["test_r2"].mean()})
pd.DataFrame(rows).round(3)

## 4. 랜덤포레스트 최종 평가와 변수 중요도 ✍️

> 변수 이름: `rf`, `importance`

In [ ]:
# 힌트: RandomForestRegressor(n_estimators=200, random_state=42), 모델.feature_importances_

## 5. 학습 범위 밖 예측 (외삽) 📋

시멘트 양만 100 → 700 kg/m³로 바꾸고 나머지는 중앙값으로 고정해 두 모델의 예측을 비교합니다. 학습 데이터의 시멘트 최댓값은 540입니다.

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
grid = pd.DataFrame([X_train.median()] * 61)
grid["cement"] = np.linspace(100, 700, 61)

plt.plot(grid["cement"], lr.predict(grid), label="선형회귀")
plt.plot(grid["cement"], rf.predict(grid), label="랜덤포레스트")
plt.axvline(X_train["cement"].max(), color="gray", linestyle="--", label="학습 최댓값")
plt.xlabel("시멘트 (kg/m3)")
plt.ylabel("예측 강도 (MPa)")
plt.title("학습 범위 밖에서의 예측")
plt.legend()
plt.show()

## 정리 💬

**Q1.** 세 모델의 교차검증 MAE를 비교하고, 현장에 무엇을 추천하겠습니까? 선형회귀가 나은 상황도 있을까요?

> 답:


**Q2.** 변수 중요도 1·2위는? 외삽 그래프에서 시멘트 540 kg/m³ 이후 두 모델은 어떻게 다르게 행동하나요?

> 답:
